In [14]:
import ast
import importlib.util
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
import datetime

# Install dependency before importing transformers
if importlib.util.find_spec("accelerate") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "accelerate>=1.1.0"])

from datasets import Dataset
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from dotenv import load_dotenv
import os
load_dotenv()

True

In [20]:
# Config
dataset_name = 'masked_금융 고객 상담 마스킹_202604052240.csv'
hf_token = os.getenv("HUGGINGFACE_TOKEN")
model_name = "klue/bert-base"
# Load dataset
df = pd.read_csv(f"data/{dataset_name}")

### Train

In [3]:
# masked_word 파싱 함수
def parse_masked_word(x):
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

df["masked_word"] = df["masked_word"].apply(parse_masked_word)
df.head(1)

,raw_text,masked_text,masked_word
0,"김영희 고객님, 전화번호 010-1234-5678로 연락주시면 상담 도와드리겠습니다.","[PERSON_NAME] 고객님, 전화번호 [PHONE_NUMBER]로 연락주시면 ...","[{'word': '김영희', 'variable_name': 'PERSON_NAME..."


In [4]:
# entity 칼럼 생성 함수
def find_span(raw_text, word, used_spans):
    start_search = 0
    
    while True:
        start = raw_text.find(word, start_search)
        if start == -1:
            raise ValueError(f"'{word}' 를 raw_text에서 찾을 수 없습니다.\nraw_text: {raw_text}")
        
        end = start + len(word)
        
        overlap = False
        for s, e in used_spans:
            if not (end <= s or start >= e):
                overlap = True
                break
        
        if not overlap:
            return start, end
        
        start_search = start + 1


def make_entity(raw_text, masked_word_list):
    entities = []
    used_spans = []
    
    for item in masked_word_list:
        word = item["word"]
        label = item["variable_name"]
        start, end = find_span(raw_text, word, used_spans)
        used_spans.append((start, end))
        
        entities.append({
            "text": word,
            "label": label,
            "start_raw": start,
            "end_raw": end
        })
    
    entities = sorted(entities, key=lambda x: x["start_raw"])
    return entities


df["entity"] = df.apply(lambda row: make_entity(row["raw_text"], row["masked_word"]), axis=1)
df[["raw_text", "entity"]].head(1)

,raw_text,entity
0,"김영희 고객님, 전화번호 010-1234-5678로 연락주시면 상담 도와드리겠습니다.","[{'text': '김영희', 'label': 'PERSON_NAME', 'star..."


In [5]:
df['entity'].loc[0]

[{'text': '김영희', 'label': 'PERSON_NAME', 'start_raw': 0, 'end_raw': 3},
 {'text': '010-1234-5678',
  'label': 'PHONE_NUMBER',
  'start_raw': 14,
  'end_raw': 27}]

In [6]:
# bio_tagging 칼럼 생성 함수
def make_bio_tagging(raw_text, entities):
    tags = ["O"] * len(raw_text)
    
    for ent in entities:
        start = ent["start_raw"]
        end = ent["end_raw"]
        label = ent["label"]
        
        tags[start] = f"B-{label}"
        for i in range(start + 1, end):
            tags[i] = f"I-{label}"
    
    return tags


df["bio_tagging"] = df.apply(lambda row: make_bio_tagging(row["raw_text"], row["entity"]), axis=1)
df[["raw_text", "bio_tagging"]].head(1)

,raw_text,bio_tagging
0,"김영희 고객님, 전화번호 010-1234-5678로 연락주시면 상담 도와드리겠습니다.","[B-PERSON_NAME, I-PERSON_NAME, I-PERSON_NAME, ..."


In [7]:
print(df['bio_tagging'].loc[0])

['B-PERSON_NAME', 'I-PERSON_NAME', 'I-PERSON_NAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'I-PHONE_NUMBER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [8]:
# 라벨 맵 만들기
entity_types = sorted({
    ent["label"]
    for entities in df["entity"]
    for ent in entities
})

label_list = ["O"]
for ent_type in entity_types:
    label_list.append(f"B-{ent_type}")
    label_list.append(f"I-{ent_type}")

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label_list)

['O', 'B-ACCOUNT_BALANCE', 'I-ACCOUNT_BALANCE', 'B-ACCOUNT_NUMBER', 'I-ACCOUNT_NUMBER', 'B-AUTH_CODE', 'I-AUTH_CODE', 'B-BIRTH_DATE', 'I-BIRTH_DATE', 'B-CARD_NUMBER', 'I-CARD_NUMBER', 'B-CUSTOMER_ID', 'I-CUSTOMER_ID', 'B-DRIVER_LICENSE', 'I-DRIVER_LICENSE', 'B-EMAIL', 'I-EMAIL', 'B-OTP_CODE', 'I-OTP_CODE', 'B-PASSWORD', 'I-PASSWORD', 'B-PERSON_NAME', 'I-PERSON_NAME', 'B-PHONE_NUMBER', 'I-PHONE_NUMBER', 'B-RESIDENT_ID', 'I-RESIDENT_ID', 'B-TRANSACTION_AMOUNT', 'I-TRANSACTION_AMOUNT']


In [ ]:
# Split train/valid
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df[["raw_text", "entity"]], preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df[["raw_text", "entity"]], preserve_index=False)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [10]:
# Set Label
def create_char_labels(raw_text, entities):
    char_labels = ["O"] * len(raw_text)
    
    for ent in entities:
        start = ent["start_raw"]
        end = ent["end_raw"]
        label = ent["label"]
        
        char_labels[start] = f"B-{label}"
        for i in range(start + 1, end):
            char_labels[i] = f"I-{label}"
    
    return char_labels

# Token Lable
def tokenize_and_align_labels(example):
    raw_text = example["raw_text"]
    entities = example["entity"]
    char_labels = create_char_labels(raw_text, entities)
    
    tokenized = tokenizer(
        raw_text,
        truncation=True,
        max_length=256,
        return_offsets_mapping=True
    )
    
    labels = []
    for start, end in tokenized["offset_mapping"]:
        if start == end:
            labels.append(-100)
        else:
            span_labels = char_labels[start:end]
            token_label = "O"
            for lab in span_labels:
                if lab != "O":
                    token_label = lab
                    break
            labels.append(label2id[token_label])
    
    tokenized["labels"] = labels
    return tokenized


In [11]:
# Tokenize Dataset
train_dataset = train_dataset.map(tokenize_and_align_labels)
valid_dataset = valid_dataset.map(tokenize_and_align_labels)

train_dataset = train_dataset.remove_columns(["raw_text", "entity"])
valid_dataset = valid_dataset.remove_columns(["raw_text", "entity"])

Map: 100%|██████████| 10/10 [00:00<00:00, 873.18 examples/s]


In [12]:
# Load Model
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8804.71it/s]
BertForTokenClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loa

In [25]:
# Metric function used by Trainer
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags = []
        label_tags = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id != -100:
                pred_tags.append(id2label[pred_id])
                label_tags.append(id2label[label_id])
        true_predictions.append(pred_tags)
        true_labels.append(label_tags)

    return {"f1": f1_score(true_labels, true_predictions)}

# Set Training Arguments
safe_model_name = model_name.replace("/", "_")
safe_dataset_name = dataset_name.replace(".csv", "").replace(" ", "_")
date = datetime.datetime.now().strftime("%Y%m%d%H%M")
training_args = TrainingArguments(
    output_dir=f"./model/{safe_model_name}_{safe_dataset_name}_{date}",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
 )

# Generate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)


In [ ]:
# Train
trainer.train()

In [27]:
# Results
pred_output = trainer.predict(valid_dataset)
print(pred_output.metrics)

{'test_loss': 0.2082708328962326, 'test_f1': 0.9565217391304348, 'test_runtime': 0.1359, 'test_samples_per_second': 73.584, 'test_steps_per_second': 14.717}


In [28]:
predictions = np.argmax(pred_output.predictions, axis=2)
labels = pred_output.label_ids

true_predictions = []
true_labels = []

for pred_seq, label_seq in zip(predictions, labels):
    pred_tags = []
    true_tags = []
    
    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id != -100:
            pred_tags.append(id2label[pred_id])
            true_tags.append(id2label[label_id])
    
    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

print(classification_report(true_labels, true_predictions))


                 precision    recall  f1-score   support

ACCOUNT_BALANCE       1.00      1.00      1.00         1
      AUTH_CODE       1.00      1.00      1.00         1
    CARD_NUMBER       1.00      1.00      1.00         2
    CUSTOMER_ID       1.00      1.00      1.00         3
          EMAIL       1.00      1.00      1.00         1
       PASSWORD       0.00      0.00      0.00         1
    PERSON_NAME       1.00      1.00      1.00         2
   PHONE_NUMBER       1.00      1.00      1.00         1

      micro avg       1.00      0.92      0.96        12
      macro avg       0.88      0.88      0.88        12
   weighted avg       0.92      0.92      0.92        12



                 precision    recall  f1-score   support

ACCOUNT_BALANCE       1.00      1.00      1.00         1
      AUTH_CODE       1.00      1.00      1.00         1
    CARD_NUMBER       1.00      1.00      1.00         2
    CUSTOMER_ID       1.00      1.00      1.00         3
          EMAIL       1.00      1.00      1.00         1
       PASSWORD       0.00      0.00      0.00         1
    PERSON_NAME       1.00      1.00      1.00         2
   PHONE_NUMBER       1.00      1.00      1.00         1

      micro avg       1.00      0.92      0.96        12
      macro avg       0.88      0.88      0.88        12
   weighted avg       0.92      0.92      0.92        12



c:\Users\user\miniconda3\envs\find\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


- Upload Model

In [25]:
from pathlib import Path
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Set this to True only when you really want to upload to Hub
do_push = False

# Use latest checkpoint if available
safe_model_name = model_name.replace("/", "_")
safe_dataset_name = dataset_name.replace(".csv", "").replace(" ", "_")
run_root = Path("model") / f"{safe_model_name}_{safe_dataset_name}"

run_candidates = sorted(
    [p for p in Path("model").glob(f"{safe_model_name}_{safe_dataset_name}_*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
 )

if run_candidates:
    run_root = run_candidates[0]

checkpoints = sorted(
    [p for p in run_root.glob("checkpoint-*") if p.is_dir()],
    key=lambda p: int(p.name.split("-")[-1]),
 )

model_load_dir = checkpoints[-1] if checkpoints else run_root
if not model_load_dir.exists():
    raise FileNotFoundError(f"모델 경로가 존재하지 않습니다: {model_load_dir}")

# IMPORTANT: repo_id must not end with '/'
repo_id = "GAYEON6423/Compliance_v1"
if repo_id.endswith("/"):
    repo_id = repo_id.rstrip("/")

if not hf_token:
    raise ValueError("HUGGINGFACE_TOKEN이 비어 있습니다. .env 또는 환경변수를 확인하세요.")

print("load dir:", model_load_dir)
print("repo id:", repo_id)
login(token=hf_token)

model = AutoModelForTokenClassification.from_pretrained(str(model_load_dir))
tokenizer = AutoTokenizer.from_pretrained(str(model_load_dir))

if do_push:
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    print("Upload complete")
else:
    print("Loaded successfully (upload skipped). Set do_push=True to upload.")

load dir: model\klue_bert-base_masked_금융_고객_상담_마스킹_202604052240_202604052323\checkpoint-250
repo id: GAYEON6423/Compliance_v1


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7199.87it/s]

Loaded successfully (upload skipped). Set do_push=True to upload.


### Inference

In [6]:
def predict_entities(model, text):
    model.eval()
    
    encoded = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    
    offset_mapping = encoded.pop("offset_mapping")[0].tolist()
    
    with torch.no_grad():
        outputs = model(**encoded)
    
    pred_ids = outputs.logits.argmax(dim=-1)[0].tolist()
    
    entities = []
    current_entity = None
    
    for pred_id, (start, end) in zip(pred_ids, offset_mapping):
        if start == end:
            continue
        
        label = id2label[pred_id]
        
        if label == "O":
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None
            continue
        
        tag, ent_type = label.split("-", 1)
        
        if tag == "B":
            if current_entity is not None:
                entities.append(current_entity)
            current_entity = {
                "text": text[start:end],
                "label": ent_type,
                "start_raw": start,
                "end_raw": end
            }
        
        elif tag == "I":
            if current_entity is not None and current_entity["label"] == ent_type:
                current_entity["text"] = text[current_entity["start_raw"]:end]
                current_entity["end_raw"] = end
            else:
                current_entity = {
                    "text": text[start:end],
                    "label": ent_type,
                    "start_raw": start,
                    "end_raw": end
                }
    
    if current_entity is not None:
        entities.append(current_entity)
    
    return entities


In [7]:
# 마스킹 문장 생성
def make_masked_text(text, entities):
    result = []
    last_end = 0
    
    for ent in sorted(entities, key=lambda x: x["start_raw"]):
        start = ent["start_raw"]
        end = ent["end_raw"]
        label = ent["label"]
        
        result.append(text[last_end:start])
        result.append(f"[{label}]")
        last_end = end
    
    result.append(text[last_end:])
    return "".join(result)


In [8]:
# Load trained model and tokenizer for inference
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_load_dir = "model\klue_bert-base_masked_금융_고객_상담_마스킹_202604052240_202604052323\checkpoint-250"
print("load dir:", model_load_dir)

tokenizer = AutoTokenizer.from_pretrained(model_load_dir)
model = AutoModelForTokenClassification.from_pretrained(model_load_dir)
id2label = model.config.id2label
label2id = model.config.label2id

print("num labels:", model.config.num_labels)

load dir: model\klue_bert-base_masked_금융_고객_상담_마스킹_202604052240_202604052323\checkpoint-250


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8433.53it/s]

num labels: 29


In [12]:
# 추론 테스트
sample_text = "홍길동 고객님의 전화번호는 010-1234-5678이고 계좌번호는 123-456-789012입니다."   
pred_entities = predict_entities(model, sample_text)
pred_masked_text = make_masked_text(sample_text, pred_entities)
print("원문:", sample_text)
print("예측 entity:", pred_entities)
print("마스킹 결과:", pred_masked_text)        

원문: 홍길동 고객님의 전화번호는 010-1234-5678이고 계좌번호는 123-456-789012입니다.
예측 entity: [{'text': '홍길동', 'label': 'PERSON_NAME', 'start_raw': 0, 'end_raw': 3}, {'text': '010-1234-5678', 'label': 'PHONE_NUMBER', 'start_raw': 15, 'end_raw': 28}, {'text': '123-456-789012', 'label': 'ACCOUNT_NUMBER', 'start_raw': 37, 'end_raw': 51}]
마스킹 결과: [PERSON_NAME] 고객님의 전화번호는 [PHONE_NUMBER]이고 계좌번호는 [ACCOUNT_NUMBER]입니다.
